# 01 · Profiling the raw data

Goal: find every problem in the raw files **before** changing anything. Findings go in the issues log at the bottom.

> All data is simulated.

## Load the files

In [ ]:
import pandas as pd
import os

RAW = "../01_raw/"

scrap  = pd.read_csv(RAW + "scrap_log_handwritten.csv", dtype=str)
prod   = pd.read_csv(RAW + "production_export_mes.csv", dtype=str)
skus   = pd.read_csv(RAW + "sku_master.csv")
assets = pd.read_csv(RAW + "asset_master.csv")
lots   = pd.read_csv(RAW + "resin_lots.csv")

for name, df in [("scrap", scrap), ("prod", prod), ("skus", skus), ("assets", assets), ("lots", lots)]:
    print(f"{name:7} {df.shape[0]:>6} rows  {df.shape[1]} columns")

## Confirm the raw files are the originals

Raw files must never be opened and saved in Excel: it changes date formats and adds empty columns.

In [ ]:
expected = {
    "asset_master.csv": 267,
    "production_export_mes.csv": 129536,
    "resin_lots.csv": 1731,
    "scrap_log_handwritten.csv": 404454,
    "sku_master.csv": 840,
}
for name, size in expected.items():
    actual = os.path.getsize(RAW + name)
    print(f"{name:28} {actual:>7}  {'OK' if actual == size else 'DIFFERENT - not the original'}")

## First look

In [ ]:
display(prod.head(10))
display(scrap.head(20))

---
# Part A · Production export

### A1. Expected vs actual row count

In [ ]:
weeks, days_per_week, machines, shifts = 13, 6, 8, 3
expected_rows = weeks * days_per_week * machines * shifts
print("Expected rows:", expected_rows)
print("Actual rows:  ", len(prod))

dupes = prod.duplicated(subset=["shift_start", "asset_id"]).sum()
print("Duplicate machine-shift rows:", dupes)

### A2. Convert types and check blanks

In [ ]:
num_cols = ["good_qty", "run_min", "changeover_min", "downtime_min", "changeover_flag"]
for c in num_cols:
    prod[c] = pd.to_numeric(prod[c], errors="coerce")

prod["shift_start"] = pd.to_datetime(prod["shift_start"], errors="coerce")
prod["shift_end"]   = pd.to_datetime(prod["shift_end"], errors="coerce")

print(prod.dtypes, "\n")
print("Blank values per column:")
print(prod.isna().sum())

In [ ]:
display(prod[prod["good_qty"].isna()])

### A3. Do the minutes add up to 480?

In [ ]:
prod["total_min"] = prod["run_min"] + prod["changeover_min"] + prod["downtime_min"]
print(prod["total_min"].value_counts().head(10))
display(prod[prod["total_min"] != 480].head(10))

### A4. Changeover flag vs changeover minutes

In [ ]:
print(pd.crosstab(prod["changeover_flag"], prod["changeover_min"] > 0,
                  rownames=["flag"], colnames=["has changeover minutes"]))

### A5. Is each product made on the right machine?

In [ ]:
check = prod.merge(skus[["sku", "asset_id"]], on="sku", how="left", suffixes=("", "_master"))
print("SKUs not found in sku_master:", check["asset_id_master"].isna().sum())
print("Rows where the machine doesn't match sku_master:",
      (check["asset_id"] != check["asset_id_master"]).sum())
print("Rows before and after join:", len(prod), len(check))

### A6. Date range and plausibility

In [ ]:
print("From", prod["shift_start"].min(), "to", prod["shift_start"].max())
print("Days of week present:", sorted(prod["shift_start"].dt.day_name().unique()), "\n")

display(prod.groupby("asset_id")["good_qty"].describe().round(0))
display(prod.groupby(["asset_id", "sku"])["good_qty"].describe().round(0))

---
# Part B · Scrap log

In [ ]:
print(scrap.isna().sum(), "\n")

for col in ["shift", "machine", "scrap_type", "entered_by"]:
    print(scrap[col].value_counts(dropna=False), "\n")

match = scrap["machine"].isin(assets["floor_name"]).mean()
print(f"Machine names matching asset_master: {match:.1%}")

display(scrap.groupby("entered_by").sample(5, random_state=1))

---
# Issues log

| # | File | Column | Problem | Rows | Plan |
|---|---|---|---|---|---|
| 0 | all | — | Raw files were once saved in Excel (dates changed, empty column added) | — | Keep originals untouched; check file sizes |
| 1 | production | good_qty | Blank on 12 shifts across 7 machines; machines were running (340–473 min) | 12 | Estimate from std rate × run time, flag as estimated |
| 2 | scrap log | date | 4 date formats, one per person | ? | Convert each person's format separately |

**Checks that passed (production):** complete rows (1,872, one per machine per shift, no duplicates) · minutes add to 480 on every shift · changeover flag matches minutes · every SKU found and made on its assigned machine · output plausible once split by product (compare machines in kg, not units).

**Note:** night shift starts 22:00 and is recorded under its start date.